# Chapter 4: Transfer Learning for Text Classification
**Module 03 – Deep Learning for Text with PyTorch**

> *Instructor: Shubham Jain, Data Scientist*

## 4.1 What Is Transfer Learning?

Transfer learning means reusing **knowledge from a model trained on one task** for a related new task.

**Benefits:**
- ⏱️ Saves training time (no training from scratch)
- 📉 Requires much less labeled data
- 🎯 Often achieves better performance

**Analogy:** An English teacher who starts teaching History already has language skills — they don't start from zero.

## 4.2 Mechanics of Transfer Learning

```
Pre-trained Model  →  Freeze base layers  →  Add task-specific head  →  Fine-tune
```

1. **Load** a large pre-trained model (e.g., BERT)
2. **Freeze** the base layers (keep their weights)
3. **Add** a new classification head for your task
4. **Fine-tune** on your smaller dataset

## 4.3 BERT — The Pre-trained Model

**BERT** = Bidirectional Encoder Representations from Transformers

| Feature | Detail |
|---|---|
| Architecture | Transformer encoder (multiple layers) |
| Training | Pre-trained on massive text corpora |
| Bidirectional | Sees both left and right context simultaneously |
| Variants | `bert-base-uncased` (12 layers), `bert-large-uncased` (24 layers) |

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# Sample data
texts  = ["I love this product!", "This is terrible.",
          "Amazing experience!", "Not my cup of tea."]
labels = [1, 0, 1, 0]  # 1=positive, 0=negative

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2  # Binary classification
)

print("BERT model loaded!")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

# Tokenize
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
    max_length=32
)
inputs["labels"] = torch.tensor(labels)

print("\nTokenized keys:", list(inputs.keys()))
print("input_ids shape:", inputs['input_ids'].shape)

## 4.4 Fine-tuning BERT

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

# Fine-tuning loop
model.train()
for epoch in range(3):
    outputs = model(**inputs)
    loss = outputs.loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/3 | Loss: {loss.item():.4f}")

## 4.5 Evaluating on New Text

In [ ]:
model.eval()

test_text = "I had an awesome day!"
input_eval = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

with torch.no_grad():
    outputs_eval = model(**input_eval)

probs = torch.nn.functional.softmax(outputs_eval.logits, dim=-1)
predicted_class = torch.argmax(probs).item()
label_map = {0: 'Negative', 1: 'Positive'}

print(f"Text: '{test_text}'")
print(f"Probabilities: Negative={probs[0,0]:.3f}, Positive={probs[0,1]:.3f}")
print(f"Prediction: {label_map[predicted_class]}")

## Summary

| Concept | Key Point |
|---|---|
| Transfer learning | Reuse knowledge from large pre-trained models |
| BERT | Bidirectional transformer pre-trained on massive text |
| Fine-tuning | Update model weights on task-specific data |
| HuggingFace | Library providing pre-trained models and tokenizers |